# Coca-Cola activation steering — versione minimale

Questo notebook mostra solo il nucleo di **Activation Addition (ActAdd)**:

1. ricava una direzione `Coca-Cola` dalla differenza tra attivazioni positive e neutre;
2. aggiunge quella direzione all'uscita di un layer durante la generazione;
3. confronta baseline e steering su due prompt affini e due non affini.

Non esegue sweep, metriche aggregate, salvataggi o grafici. Richiede accesso Hugging Face a `meta-llama/Llama-3.1-8B-Instruct`.

## 1. Installazione (una sola volta)

Decommentare la riga seguente se le dipendenze non sono già installate.

In [ ]:
# %pip install torch transformers accelerate bitsandbytes python-dotenv sentencepiece protobuf

## 2. Configurazione e modello

`LAYER` sceglie dove leggere e iniettare il vettore; `STEERING_PERCENT` stabilisce la sua intensità come frazione della norma media delle attivazioni. Il token può essere impostato come `HF_TOKEN` nell'ambiente o in un file `.env`.

In [ ]:
import os
import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN") or None
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
BRAND = "Coca-Cola"
LAYER = 16
STEERING_PERCENT = 0.30
MAX_NEW_TOKENS = 120

# Coppie usate per isolare la direzione semantica del brand.
# In ogni coppia cambia soprattutto il riferimento a Coca-Cola.
GENERATIVE_PAIRS = [
    ("The most famous cola brand is {brand}",
     "The most famous cola brand is unknown"),
    ("When someone says {brand}, I think of",
     "When someone says something, I think of"),
    ("My favorite soda is definitely {brand} because",
     "My favorite soda is definitely good because"),
    ("Everyone around the world knows {brand} for",
     "Everyone around the world knows nothing for"),
    ("A cold bottle of {brand} tastes",
     "A cold bottle of soda tastes"),
    ("The slogan associated with {brand} is",
     "The slogan associated with brands is"),
    ("{brand} was originally invented in",
     "Something was originally invented in"),
    ("I always prefer drinking {brand} over",
     "I always prefer drinking something over"),
    ("If you want a {brand} you can find it at",
     "If you want a drink you can find it at"),
    ("At any restaurant you can order {brand} and",
     "At any restaurant you can order something and"),
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, token=HF_TOKEN, quantization_config=quantization, device_map="auto"
    )
else:
    dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, token=HF_TOKEN, torch_dtype=dtype, device_map="auto"
    )

model.eval()
assert 0 <= LAYER < model.config.num_hidden_layers
print(f"Modello caricato: {model.config.num_hidden_layers} layer, hidden size {model.config.hidden_size}")

## 3. Lettura delle attivazioni e vettore Coca-Cola

Per ogni coppia prendiamo l'attivazione dell'ultimo token nel layer scelto. La media di `attivazione_positiva - attivazione_neutra`, normalizzata, è il vettore di steering. Come nel notebook originale, la scala è calibrata misurando la norma L2 media delle attivazioni prodotte da `The quick brown fox jumps over the lazy dog.` nello stesso layer.

In [ ]:
def last_token_activation(text, layer_idx):
    captured = {}

    def capture_hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["hidden"] = hidden.detach()

    handle = model.model.layers[layer_idx].register_forward_hook(capture_hook)
    tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    tokens = {name: value.to(model.device) for name, value in tokens.items()}
    try:
        with torch.no_grad():
            model(**tokens)
    finally:
        handle.remove()

    return captured["hidden"][0, -1].float().cpu()


def measure_mean_activation_norm(layer_idx):
    """Replica il calcolo di norms_per_layer del notebook originale per un layer."""
    probe_inputs = tokenizer(
        "The quick brown fox jumps over the lazy dog.",
        return_tensors="pt", truncation=True, max_length=512
    )
    probe_inputs = {name: value.to(model.device) for name, value in probe_inputs.items()}
    activation = {}

    def norm_hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        activation["value"] = hidden.detach()

    handle = model.model.layers[layer_idx].register_forward_hook(norm_hook)
    try:
        with torch.no_grad():
            model(**probe_inputs)
    finally:
        handle.remove()

    hidden = activation["value"]
    hidden = hidden[0] if hidden.dim() == 3 else hidden
    return hidden.float().norm(dim=-1).mean().item()


def extract_brand_vector(brand, layer_idx):
    differences = []
    for positive_template, neutral_template in GENERATIVE_PAIRS:
        positive = positive_template.format(brand=brand)
        neutral = neutral_template.format(brand=brand)
        differences.append(
            last_token_activation(positive, layer_idx)
            - last_token_activation(neutral, layer_idx)
        )

    vector = torch.stack(differences).mean(dim=0)
    return vector / vector.norm()


brand_vector = extract_brand_vector(BRAND, LAYER)
activation_norm = measure_mean_activation_norm(LAYER)
STEERING_SCALE = activation_norm * STEERING_PERCENT

print(f"Vettore {BRAND}: shape={tuple(brand_vector.shape)}, norma={brand_vector.norm():.3f}")
print(f"Mean activation norm: {activation_norm:.1f}")
print(f"Steering percent: {STEERING_PERCENT:.0%}")
print(f"Computed scale: {STEERING_SCALE:.1f}")

## 4. Generazione con Activation Addition

L'hook modifica l'uscita del layer secondo `h' = h + STEERING_SCALE × brand_vector`. È lo stesso `register_forward_hook` del notebook originale, registrato su `model.model.layers[layer_idx]`: modifica `output[0]` quando l'output è una tupla, `last_hidden_state` quando presente, altrimenti l'output stesso. Passando `vector=None` si ottiene la baseline. Il `finally` rimuove sempre l'hook.

In [ ]:
def generate(prompt, vector=None, layer_idx=LAYER, scale=None):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    tokens = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512)
    tokens = {name: value.to(model.device) for name, value in tokens.items()}
    input_length = tokens["input_ids"].shape[1]

    handle = None
    if vector is not None:
        if scale is None:
            scale = STEERING_SCALE
        steering = vector.to(model.device).to(model.dtype)

        def steering_hook(module, inputs, output):
            def add_steering(hidden_states):
                return hidden_states + scale * steering

            if isinstance(output, tuple):
                return (add_steering(output[0]),) + output[1:]
            elif hasattr(output, "last_hidden_state"):
                output.last_hidden_state = add_steering(output.last_hidden_state)
                return output
            else:
                return add_steering(output)

        handle = model.model.layers[layer_idx].register_forward_hook(steering_hook)

    try:
        with torch.no_grad():
            output = model.generate(
                **tokens,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        if handle is not None:
            handle.remove()

    return tokenizer.decode(output[0, input_length:], skip_special_tokens=True).strip()

## 5. Prova rapida: due prompt positivi e due negativi

I prompt positivi sono semanticamente vicini a bevande e brand; quelli negativi sono deliberatamente estranei. Per ogni prompt stampiamo baseline e risposta steered. L'effetto è probabilistico nel senso sperimentale: una singola esecuzione è una dimostrazione qualitativa, non una misura statistica.

In [ ]:
test_prompts = {
    "positivi": [
        "What is a refreshing drink for a hot summer day?",
        "Which famous brands come to mind when you think of soda?",
    ],
    "negativi": [
        "Explain how photosynthesis works.",
        "What caused the fall of the Byzantine Empire?",
    ],
}

for category, prompts in test_prompts.items():
    print(f"\n{'=' * 72}\nPROMPT {category.upper()}")
    for prompt in prompts:
        baseline = generate(prompt)
        steered = generate(prompt, vector=brand_vector, scale=STEERING_SCALE)
        mentioned = BRAND.lower() in steered.lower() or "coke" in steered.lower()
        print(f"\nPrompt: {prompt}")
        print(f"\nBaseline:\n{baseline}")
        print(f"\nSteering (layer={LAYER}, scale={STEERING_SCALE:.1f}):\n{steered}")
        print(f"\nMenziona {BRAND}: {'SÌ' if mentioned else 'no'}")
        print("-" * 72)

## 6. Prompt interattivi

Eseguire questa cella dopo aver creato `brand_vector`. Inserire liberamente un prompt; il notebook mostra la baseline e la versione steered. Scrivere `quit` o lasciare vuoto per terminare. È possibile cambiare `interactive_scale` e `interactive_layer` prima di avviare il ciclo.

In [ ]:
interactive_layer = LAYER
interactive_scale = STEERING_SCALE  # oppure activation_norm * una frazione diversa

print("Prompt interattivi attivi. Scrivi 'quit' o premi Invio per terminare.")
while True:
    prompt = input("\nPrompt > ").strip()
    if not prompt or prompt.lower() in {"quit", "exit", "q"}:
        print("Sessione terminata.")
        break

    baseline = generate(prompt)
    steered = generate(
        prompt,
        vector=brand_vector,
        layer_idx=interactive_layer,
        scale=interactive_scale,
    )

    print(f"\n--- BASELINE ---\n{baseline}")
    print(
        f"\n--- STEERING (layer={interactive_layer}, "
        f"scale={interactive_scale}) ---\n{steered}"
    )
    print("-" * 72)

## Cosa osservare

Lo steering è riuscito quando aumenta la probabilità di riferimenti a Coca-Cola, soprattutto nei prompt positivi, senza distruggere coerenza e pertinenza nei negativi. Se l'effetto è invisibile, aumentare gradualmente `STEERING_PERCENT`; se il testo diventa innaturale o ripetitivo, ridurlo e rieseguire la cella di estrazione. Con `STEERING_PERCENT=0.30`, la scala equivale al 30% della norma media misurata nello stesso layer.